<!-- NB_DOC_INTRO_V1 -->
# RAG end-to-end avec vector DB + extraction PDF

> 📚 **Doc thematique** : [docs/07_BDD_DE.md](docs/07_BDD_DE.md) (Bases de donnees / Data Engineering / Web)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Pipeline RAG complet** : ingerer des PDF, chunker, embed, indexer, retrieve, generer. Ce notebook execute la version **from-scratch** (sans LangChain) avec sentence-transformers + FAISS. Pour LangChain orchestration : [NLP_LangChain_RAG.ipynb](./NLP_LangChain_RAG.ipynb).

## Plan

1. Setup
2. Documents jouet + chunking strategies
3. Embeddings (sentence-transformers)
4. Indexation FAISS
5. Retrieval + cosine similarity
6. Re-ranking (cross-encoder, concept)
7. Hybrid search (BM25 + dense + RRF)
8. Generation (concept, sans LLM appele ici)
9. Eval RAG (ragas, concept)
10. Pieges + References


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Documents jouet — corpus de phrases

In [ ]:
docs = [
    "Le RAG combine retrieval et generation pour ancrer les LLMs dans des sources.",
    "Les embeddings transforment du texte en vecteurs denses.",
    "FAISS est une lib Facebook pour la recherche de plus proches voisins.",
    "Chroma et Qdrant sont des vector databases modernes.",
    "Sentence-Transformers fine-tune BERT pour la similarite de phrases.",
    "Le chunking est crucial : trop petit perd le contexte, trop grand dilue.",
    "BM25 est un algo de retrieval lexical, baseline souvent imbattable.",
    "Re-ranking avec cross-encoder boost typiquement de 10-15% NDCG.",
    "Python est un langage interprete de haut niveau.",
    "Les LLMs hallucinent — RAG attenue ce probleme via citations.",
    "Hybrid search combine BM25 et dense pour le meilleur recall.",
    "PyTorch est utilise pour entrainer la plupart des LLMs en 2024.",
    "Le fine-tuning avec PEFT/LoRA permet d'adapter un LLM 7B sur GPU consumer.",
    "ChromaDB est integre nativement avec LangChain.",
    "Qdrant supporte le filtrage metadata complexe avec payload.",
]
print(f"Corpus : {len(docs)} documents")

## 2. Embeddings

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    print("Loading model (DL ~80 MB premiere fois)...")
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    print(f"Dim embeddings : {embedder.get_sentence_embedding_dimension()}")

    # Encoder le corpus
    doc_embs = embedder.encode(docs, show_progress_bar=False)
    print(f"Shape embeddings : {doc_embs.shape}")
    ST_OK = True
except ImportError:
    print("sentence-transformers pas installe")
    ST_OK = False

## 3. Indexation FAISS

In [ ]:
if ST_OK:
    import faiss
    # Normaliser pour cosine via inner product
    doc_embs_norm = doc_embs / np.linalg.norm(doc_embs, axis=1, keepdims=True)
    index = faiss.IndexFlatIP(doc_embs_norm.shape[1])
    index.add(doc_embs_norm.astype("float32"))
    print(f"Index contains : {index.ntotal} docs")
else:
    print("SKIP")

## 4. Retrieval

In [ ]:
if ST_OK:
    def search(query, k=3):
        q_emb = embedder.encode([query])
        q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
        scores, ids = index.search(q_emb.astype("float32"), k=k)
        return [(docs[i], scores[0][j]) for j, i in enumerate(ids[0])]

    queries = [
        "Comment construire un RAG ?",
        "Quelle base vectorielle utiliser ?",
        "Anti-hallucination pour LLM ?",
    ]
    for q in queries:
        print(f"\nQuery : {q}")
        for doc, sc in search(q, k=3):
            print(f"  [{sc:.3f}] {doc}")
else:
    print("SKIP")

## 5. Re-ranking avec cross-encoder (concept)

Un **cross-encoder** encode `(query, doc)` ensemble → score plus precis mais O(N) au runtime. Pattern : bi-encoder pour top-100, cross-encoder pour rerank top-10.

In [ ]:
if ST_OK:
    try:
        from sentence_transformers import CrossEncoder
        reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

        query = "Comment construire un RAG ?"
        # Top 5 du bi-encoder
        bi_top = [doc for doc, _ in search(query, k=5)]
        # Re-rank
        pairs = [(query, d) for d in bi_top]
        cross_scores = reranker.predict(pairs)
        reranked = sorted(zip(bi_top, cross_scores), key=lambda x: -x[1])

        print(f"\nRe-ranking pour : {query}")
        for d, s in reranked:
            print(f"  [{s:+.3f}] {d}")
    except Exception as e:
        print(f"Cross-encoder skip : {e}")
else:
    print("SKIP")

## 6. Hybrid search — BM25 + dense + RRF

In [ ]:
try:
    from rank_bm25 import BM25Okapi

    tokenized = [d.lower().split() for d in docs]
    bm25 = BM25Okapi(tokenized)

    query = "vector database"
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranks = sorted(range(len(docs)), key=lambda i: -bm25_scores[i])

    if ST_OK:
        q_emb = embedder.encode([query])
        q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
        dense_scores, dense_ids = index.search(q_emb.astype("float32"), k=len(docs))
        dense_ranks = dense_ids[0].tolist()

        # Reciprocal Rank Fusion
        def rrf(rankings_lists, k=60):
            scores = {}
            for rk_list in rankings_lists:
                for rank, doc_id in enumerate(rk_list, 1):
                    scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
            return sorted(scores.items(), key=lambda x: -x[1])

        fused = rrf([bm25_ranks, dense_ranks])
        print(f"Hybrid (BM25 + dense + RRF) pour : {query}")
        for doc_id, score in fused[:5]:
            print(f"  [RRF={score:.4f}] {docs[doc_id]}")
except ImportError:
    print("rank_bm25 pas installe : uv add --group vector rank-bm25")

## 7. Generation (concept, sans LLM appele ici)

Une fois les top-k docs recuperes, on **construit un prompt** :

```python
def build_rag_prompt(query, retrieved_docs):
    context = "\n\n".join(f"[{i+1}] {d}" for i, d in enumerate(retrieved_docs))
    return f'''Tu es un assistant qui repond UNIQUEMENT en t'appuyant sur le contexte fourni.
Si la reponse n'est pas dans le contexte, dis "Je ne sais pas".

Contexte :
{context}

Question : {query}

Reponse :'''

# Avec LLM local (Ollama) ou API (OpenAI / Anthropic)
# from openai import OpenAI
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# r = client.chat.completions.create(model="llama3.2:3b",
#                                     messages=[{"role":"user","content": build_rag_prompt(q, top_docs)}])
```

Pour le pipeline complet avec LangChain : [NLP_LangChain_RAG.ipynb](./NLP_LangChain_RAG.ipynb).

## 8. Eval RAG — metriques `ragas`

| Metrique | Description |
|---|---|
| **Faithfulness** | Reponse ancree dans le contexte (anti-hallucination) |
| **Answer relevancy** | Reponse repond a la question |
| **Context precision** | Chunks recuperes sont pertinents |
| **Context recall** | Tous les chunks pertinents sont recuperes |

Cf. [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) section 13.

## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Chunks trop petits (200 chars) | 500-1000 chars + overlap 10-20% |
| Pas d'overlap | Perd les debordements d'info |
| Top-k = 50 sans re-ranking | Context blow-up |
| Pas de 'say I don't know' dans prompt | Hallucinations |
| Embeddings raw BERT pour retrieval | Sentence-Transformers est specifique a la similarite |
| Pas tester recall sur dataset gold | Pas de mesure de qualite |
| Hybrid sans RRF tuning | k=60 est un bon default mais a tester |
| Persister Chroma sans `persist_directory` | Perte au restart |


## References

### Documentation
- FAISS : https://github.com/facebookresearch/faiss
- Sentence-Transformers : https://www.sbert.net/
- ragas : https://docs.ragas.io/
- MTEB leaderboard : https://huggingface.co/spaces/mteb/leaderboard

### Voir aussi
- [BDD_Vectorielles.ipynb](BDD_Vectorielles.ipynb)
- [NLP_LangChain_RAG.ipynb](NLP_LangChain_RAG.ipynb)
- [NLP_Recherche_d_informations.ipynb](NLP_Recherche_d_informations.ipynb)
- [AI_Prompt_Engineering.ipynb](AI_Prompt_Engineering.ipynb)
